# Exercise 3.5

This exercise looks at an example of using Belief Propagation using a student model adapted from Koller & Friedman, Probabilistic Graphical Models, 2009.
The graphical model and CPDs are shown below, together with a description of the domain.



<figure role="group">
  <img src="../images/Fig4.37.png" alt="Model visualisation" />
</figure>


Koller & Friedman, Probabilistic Graphical Models, 2009, p.53f.


The model describes how a student’s grade, depends not only on their intelligence but also on the difficulty of the course, represented by a random variable {easy(0), hard(1)}.
The student asks their professor for a recommendation letter. The professor is absentminded and never remembers the names of their students. They can only look at the student's grade, and write a letter based on that information alone. The quality of the letter is a random variable {strong=0, weak=1}. The actual quality of the letter depends stochastically on the grade. (It can vary depending on how stressed the professor is and the quality of the coffee they had that morning.)
We therefore have five random variables in this domain: the student’s intelligence (intel), the course difficulty (diff), the grade (grade), the student’s SAT score (SAT), and the quality of the recommendation letter (letter). All of the variables except grade and intel are binary-valued, and grade and intell are ternary-valued. 

The course difficulty and the student’s intelligence are determined independently, and before any of the variables in the model. The student’s grade depends on both of these factors. The student’s SAT score depends only on his intelligence. The quality of the professor’s recommendation letter depends (by assumption) only on the student’s grade in the class.

The code below creates the Bayesian network and CPD tables

In [2]:
#This code is modified from the examples at pgmpy.org Copyright (c) 2013-2021 pgmpy
from pgmpy.models import BayesianModel
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import BeliefPropagation
G = BayesianModel([('diff', 'grade'), ('intel', 'grade'),
                   ('intel', 'SAT'), ('grade', 'letter')])
diff_cpd = TabularCPD('diff', 2, [[0.2], [0.8]])
intel_cpd = TabularCPD('intel', 3, [[0.5], [0.3], [0.2]])
grade_cpd = TabularCPD('grade', 3,
                       [[0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
                        [0.1, 0.1, 0.1, 0.1, 0.1, 0.1],
                        [0.8, 0.8, 0.8, 0.8, 0.8, 0.8]],
                       evidence=['diff', 'intel'],
                       evidence_card=[2, 3])
sat_cpd = TabularCPD('SAT', 2,
                     [[0.1, 0.2, 0.7],
                      [0.9, 0.8, 0.3]],
                     evidence=['intel'], evidence_card=[3])
letter_cpd = TabularCPD('letter', 2,
                        [[0.1, 0.4, 0.8],
                         [0.9, 0.6, 0.2]],
                        evidence=['grade'], evidence_card=[3])
G.add_cpds(diff_cpd, intel_cpd, grade_cpd, sat_cpd, letter_cpd)


To perform inference we first need to calculate the messages and then compute our query. In this case we are interested in whether the quality of the reference letter will be strong (0) or weak (1), for a student who is studying an easy course, has a low grade and has a low SAT score. We can use a MAP query for this.

In [3]:
bp = BeliefPropagation(G)
#bp.calibrate()
bp.map_query(variables=['letter'],
                             evidence={'diff': 0, 'grade': 0, 'SAT': 0})

{'letter': 1}

As expected this student would most likely received a weak reference.
Now try a few other queries to check that you get the expected output. 

If we wish to calcualte the probability distribtuion for a variable we can just use "query"

In [4]:

phi = bp.query(variables=['letter'],
                             evidence={'diff': 0, 'grade': 0, 'SAT': 0})

In [5]:
print(phi)

+-----------+---------------+
| letter    |   phi(letter) |
+===========+===============+
| letter(0) |        0.1000 |
+-----------+---------------+
| letter(1) |        0.9000 |
+-----------+---------------+
